# Molecular properties vs simulation performance

Correlates per-molecule cosine scores from the Franklin benchmark with
molecular descriptors: carbon number, oxygen count, TMS substitution count,
and SIMPOL functional group counts.

**Prerequisite:** run the SIMPOL cell below once (requires OpenBabel module).
After that the CSV is cached and the module is not needed again.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats

sys.path.insert(0, os.path.abspath('..'))

DATA_ROOT = os.path.abspath('../data')
SRC_ROOT  = os.path.abspath('../src')

GREY_TEXT = '#4C4C4C'
METHOD_PALETTE = {
    'QCxMS':  '#5A99D3',
    'QCxMS2': '#4C4C4C',
    'CFMID':  '#A6A6A6',
    'NEIMS':  '#D36EA5',
}
QCXMS_CANONICAL = 'QCxMS_10_ps'
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors as RDDescriptors
    _RDKIT = True
except ImportError:
    _RDKIT = False

_inter_reg = False
for _fp in ['/tmp/inter_fonts/Inter-Regular.ttf', '/tmp/inter_fonts/Inter-Bold.ttf']:
    if Path(_fp).exists():
        fm.fontManager.addfont(_fp)
        _inter_reg = True
_PAPER_FONT = ['Inter', 'Nimbus Sans', 'DejaVu Sans'] if _inter_reg else ['Nimbus Sans', 'DejaVu Sans']
C1, C2, C3 = '#5A99D3', '#D36EA5', '#7BC8A4'

# ── Output modes ──────────────────────────────────────────────────────────────
PAPER_MODE = True
PRES_MODE  = False  # set True separately to generate pres outputs
# ── Dataset registry ──────────────────────────────────────────────────────────
_fp = '../data/processed/franklin'   # Franklin and Franklin TMS share one SIMPOL CSV

DATASETS = [
    {
        "name":             "franklin_tms",
        "label":            "Franklin TMS",
        "simpol_csv":       f"{_fp}/franklin_SIMPOL_benchmark.csv",
        "dataset_csv":      f"{_fp}/dataset_unique.csv",
        "deriv_csv":        "../data/processed/franklin_tms/franklin_tms.csv",
        "simpol_tms_csv":   "../data/processed/franklin_tms/franklin_tms_SIMPOL_benchmark.csv",
        "smiles_col":       "SMILES",
        "tms_col":          "TMS",
        "sim_dir":          "../data/simulation_results/franklin_tms",
        "reports_dir":      "../reports/franklin_tms/explore",
        "paper_dir":        "../reports/franklin_tms/paper",
        "pres_dir":         "../reports/franklin_tms/pres",
        "include_tms_plot": True,
        "hist_color":       "#5A99D3",
        # Molecules where the automated TMS assignment disagrees with the reference
        # (ref TMS=0 but script added TMS groups). The TMS-derivatised simulation
        # results remain on disk; for analysis we substitute underivatised Franklin scores.
        "tms_mismatch": {
            "mol_indices":       ["0007", "0026", "0058"],
            "fallback_sim_dir":  "../data/simulation_results/franklin",
        },
    },
    {
        "name":             "franklin",
        "label":            "Franklin",
        "simpol_csv":       f"{_fp}/franklin_SIMPOL_benchmark.csv",
        "dataset_csv":      f"{_fp}/dataset_unique.csv",
        "smiles_col":       "SMILES",
        "tms_col":          "TMS",
        "sim_dir":          "../data/simulation_results/franklin",
        "reports_dir":      "../reports/franklin/explore",
        "paper_dir":        "../reports/franklin/paper",
        "pres_dir":         "../reports/franklin/pres",
        "include_tms_plot": False,
        "hist_color":       "#D36EA5",
    },
    {
        "name":             "ucb_globes_tracers",
        "label":            "UCB-GLOBES tracers",
        "simpol_csv":       "../data/processed/ucb_globes_tracers/ucb_globes_tracers_orig_SIMPOL_benchmark.csv",
        "simpol_tms_csv":   "../data/processed/ucb_globes_tracers/ucb_globes_tracers_SIMPOL_benchmark.csv",
        "dataset_csv":      "../data/processed/ucb_globes_tracers/ucb_globes_tracers.csv",
        "smiles_col":       "Original_SMILES",
        "smiles_tms_col":   "Modified_SMILES",
        "tms_col":          "Total_Replacements",
        "sim_dir":          "../data/simulation_results/ucb_globes_tracers",
        "reports_dir":      "../reports/ucb_globes_tracers/explore",
        "paper_dir":        "../reports/ucb_globes_tracers/paper",
        "pres_dir":         "../reports/ucb_globes_tracers/pres",
        "include_tms_plot": True,
        "hist_color":       "#7BC8A4",
    },
]

## 1. Generate SIMPOL descriptors (run once)

In [ ]:
# Generate SIMPOL descriptors for any dataset that does not have a cached CSV yet.
# Requires OpenBabel — skip datasets that already have the output file.
RERUN = True # set True to force regeneration of existing CSVs

_seen = set()
for ds in DATASETS:
    _sp = Path(ds['simpol_csv'])
    if _sp in _seen:
        continue          # Franklin TMS reuses the same CSV as Franklin
    _seen.add(_sp)
    if _sp.exists() and not RERUN:
        print(f'Already exists: {_sp}')
        continue
    _dc = Path(ds['dataset_csv'])
    if not _dc.exists():
        print(f'Skipping {ds["label"]}: {_dc} not found')
        continue
    _sp.parent.mkdir(parents=True, exist_ok=True)
    get_ipython().system(
        'module load openbabel && '
        f'python {SRC_ROOT}/features/generate_simpol_groups.py '
        f'-s {_dc} '
        f'-c {ds["smiles_col"]} '
        f'-g {SRC_ROOT}/aprl_ssp/SMARTSpatterns/SIMPOLgroups_sane.csv '
        f'-o {_sp}'
    )

# Also generate TMS-derivatised SIMPOL for datasets that define simpol_tms_csv.
# Uses Modified_SMILES (the derivatised structure that QCxMS actually simulates).
for ds in DATASETS:
    _sp_tms = ds.get('simpol_tms_csv')
    if not _sp_tms:
        continue
    _sp_tms = Path(_sp_tms)
    if _sp_tms.exists() and not RERUN:
        print(f'Already exists (TMS): {_sp_tms}')
        continue
    # Use deriv_csv if defined, otherwise fall back to dataset_csv
    _dc = Path(ds.get('deriv_csv') or ds['dataset_csv'])
    _smiles_tms_col = ds.get('smiles_tms_col', 'Modified_SMILES')
    if not _dc.exists():
        print(f'Skipping TMS SIMPOL for {ds["label"]}: {_dc} not found')
        continue
    _sp_tms.parent.mkdir(parents=True, exist_ok=True)
    get_ipython().system(
        'module load openbabel && '
        f'python {SRC_ROOT}/features/generate_simpol_groups.py '
        f'-s {_dc} '
        f'-c {_smiles_tms_col} '
        f'-g {SRC_ROOT}/aprl_ssp/SMARTSpatterns/SIMPOLgroups_sane.csv '
        f'-o {_sp_tms}'
    )

## 2. Analysis — all datasets

Runs load + all plots for every dataset in `DATASETS`.
Set `PAPER_MODE = True` in the CONFIG cell to also write ACS-sized PDFs to `reports/paper/`.
Outputs saved to `reports/molecular_properties_{name}/`.

In [ ]:
from src.visualization.plot_molecular_properties import run_analysis


In [ ]:
for ds in DATASETS:
    if PAPER_MODE:
        run_analysis(ds, paper=True, presentation=False)
    if PRES_MODE:
        run_analysis(ds, paper=False, presentation=True)